machine health monitoring and risk scoring (Business layer)

In [0]:

from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf, col, avg, max as _max, count, sum as _sum
from pyspark.sql.window import Window
from pyspark.sql.functions import percent_rank

predictions = spark.table("predictions")

#Extract anomaly probability
get_prob = udf(lambda v: float(v[1]), DoubleType())
pred = predictions.withColumn("anomaly_prob", get_prob(col("probability")))

#Build df_risk (machine-level health score) based on anomaly probabilities
#Machine-level aggregation. We are now thinking ""How risky is Machine 1 overall?""
df_risk = pred.groupBy("machine_id").agg(

    # average risk level
    avg("anomaly_prob").alias("avg_risk"),

    # worst-case spike (VERY IMPORTANT)
    _max("anomaly_prob").alias("max_risk"),

    # how often model predicted anomaly
    avg(col("prediction")).alias("anomaly_rate"),

    # total samples (for context)
    count("*").alias("total_records")
)

#Create HEALTH/RISK SCORE
from pyspark.sql.functions import lit

df_risk = df_risk.withColumn(
    "risk_score",
    col("avg_risk") * 0.7 +
    col("anomaly_rate") * 0.3
)


#ALERT RULES
from pyspark.sql.functions import when

w = Window.orderBy(col("risk_score").desc())

df_risk = df_risk.withColumn(
    "risk_rank",
    percent_rank().over(w)
)

#Percentile category of health of machine
from pyspark.sql.functions import when, col

quantiles = df_risk.approxQuantile("risk_score", [0.7, 0.9], 0.0)
q70 = quantiles[0]
q90 = quantiles[1]

from pyspark.sql.functions import when, col

df_risk = df_risk.withColumn(
    "status",
    when(col("risk_score") >= q90, "CRITICAL")
    .when(col("risk_score") >= q70, "WARNING")
    .otherwise("HEALTHY")
)

df_risk.select(
    "machine_id",
    "risk_score",
    "status",
    "avg_risk",
    "max_risk",
    "anomaly_rate",
    "total_records"
).orderBy("risk_score", ascending=False).show(50)



In [0]:

#top 10 riskiest machines
df_risk.orderBy(col("risk_score").desc()).select(
    "machine_id",
    "risk_score",
    "status",
    "anomaly_rate"
).show(10, False)

#top 10 safest machines
df_risk.orderBy(col("risk_score").asc()).select(
    "machine_id",
    "risk_score",
    "status",
    "anomaly_rate"
).show(10, False)

df_risk.groupBy("status").count().show()

In [0]:
pdf = df_risk.toPandas()
pdf.to_csv("/tmp/machine_risk_final.csv", index=False)
display(pdf.head())

pdf.to_csv("machine_risk_final.csv", index=False)